
- ETL 타겟 사이트 
        - https://www.starbucks.co.kr/store/store_map.do
- 수집 내용
    - 서울시 스타벅스 매장 정보 추출(Extract, Transform)
        - 매장명
        - 주소
        - 매장타입(general, reserve, generalDT)
- 저장 방식 (Load)
    - 이름.csv
- 제출 정보
    - 기한 : 2026-03-16일 24시까지
    - 형식 : 이름.ipynb
    - 이메일 : edu2.ucoc@gmail.com

In [41]:
from selenium import webdriver as wd
import time

In [42]:
driver = wd.Chrome() # 크롬 창이 열림, 이제 driver로 브라우저 조종

In [43]:
# 타겟 사이트 진입
driver.get("https://www.starbucks.co.kr/store/store_map.do")

- 지역 검색 클릭
- 시/도 정보 추출
- css selector (특정 HTML 요소 찾는 방법)

In [44]:
# css selector 사용한다는 지정 -> By, 표현식
from selenium.webdriver.common.by import By
# 지역 검색 클릭
driver.find_element(By.CSS_SELECTOR, '.loca_search > h3 > a' ).click()


In [45]:
# 시/도 정보
sidos = driver.find_elements(By.CSS_SELECTOR, '.sido_arae_box > li > a' )
len(sidos)

17

In [46]:
# 시/도 정보 중 서울만 가져오기
for sido in sidos:
    sido.click()
    # 명시적 대기
    time.sleep(10)
    break      


In [47]:
# 구/군 정보중 전체 클릭
guguns = driver.find_elements(By.CSS_SELECTOR, '.gugun_arae_box > li > a')
len(guguns)
for gugun in guguns:
    gugun.click()
    time.sleep(10)
    break # 전체만 클릭. 한번만 수행

In [48]:
import pandas as pd

stores = driver.find_elements(By.CSS_SELECTOR, '#mCSB_3_container > ul > li')

result = []
for store in stores:
    name = store.get_attribute('data-name').strip()
    
    # 매장 타입 분류
    if name[-2:] == 'DT':
        store_type = 'generalDT'
    elif name[-1:] == 'R':
        store_type = 'reserve'
    else:
        store_type = 'general'
    
    # 주소
    address_p = store.find_element(By.CSS_SELECTOR, 'p.result_details')
    address = address_p.get_attribute('textContent').strip().split('1522-3232')[0].strip()
    
    result.append({'name': name, 'address': address, 'type': store_type})
result
df = pd.DataFrame(result)
df.to_csv('성효창.csv', index=False, encoding='utf-8-sig')

In [49]:
driver.quit()